# 01 — Audit & Cleaning Dataset
### Dashboard Identifikasi dan Analisis Tren Isu Kesehatan Masyarakat di Bangkalan (SBERT-BERTopic)

**Tujuan notebook ini HANYA:**
1. Mengaudit dua dataset mentah (berita & ulasan Google Maps) tanpa mengubah apa pun.
2. Mengusulkan dan menjelaskan strategi cleaning yang cocok untuk SBERT + BERTopic.
3. Melakukan cleaning (bukan filtering relevansi, bukan pelabelan) secara reproducible.
4. Membuat kolom/flag kemungkinan relevansi (bukan keputusan final).
5. Membuat laporan kualitas SEBELUM vs SESUDAH.
6. Menyimpan dataset cleaned + audit report + log cleaning.

**Notebook ini TIDAK membuat model SBERT atau BERTopic** — itu ada di `02_sbert_bertopic.ipynb`.

Dataset mentah **tidak pernah ditimpa**. Semua kolom penting (tanggal, lokasi, rating,
publisher, kategori) tetap dipertahankan. Tidak ada stemming / stopword removal agresif
pada teks utama karena teks akan dipakai untuk semantic embedding (SBERT).


## 0. Import Library

In [ ]:
# Library dasar untuk manipulasi data
import pandas as pd
import numpy as np

# Library untuk membersihkan teks (regex, HTML, unicode)
import re
import html
import unicodedata
from datetime import datetime

# Untuk menyimpan log & audit
import json as json_lib
import os

# Opsional: deteksi bahasa (berguna untuk QA, TIDAK dipakai untuk memfilter data)
try:
    from langdetect import detect, DetectorFactory
    DetectorFactory.seed = 42
    LANGDETECT_AVAILABLE = True
except ImportError:
    LANGDETECT_AVAILABLE = False
    print("langdetect tidak terpasang, sel deteksi bahasa (opsional) akan dilewati.")

pd.set_option("display.max_colwidth", 150)
pd.set_option("display.max_columns", 50)
print("Library berhasil diimpor.")


## 1. Mount Drive & Baca Dataset

In [ ]:
# Jalankan sel ini HANYA di Google Colab.
# Jika Anda menjalankan notebook ini secara lokal (bukan Colab), lewati sel ini
# dan sesuaikan variabel path di sel berikutnya.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# --- Konfigurasi path -------------------------------------------------------
# Sesuaikan BASE_DIR dengan lokasi folder project Anda di Google Drive.
BASE_DIR = "/content/drive/MyDrive/project"

RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
CLEANED_DIR = os.path.join(BASE_DIR, "data", "cleaned")
os.makedirs(CLEANED_DIR, exist_ok=True)

NEWS_RAW_PATH = os.path.join(RAW_DIR, "berita_kesehatan_bangkalan_10tahun.csv")
REVIEW_RAW_PATH = os.path.join(RAW_DIR, "ulasan_kesehatan_lingkungan_bangkalan.csv")

print("RAW_DIR      :", RAW_DIR)
print("CLEANED_DIR  :", CLEANED_DIR)


In [ ]:
# Baca dataset mentah. encoding utf-8-sig dipakai karena file hasil scraping
# sebelumnya disimpan dengan encoding tersebut.
df_news_raw = pd.read_csv(NEWS_RAW_PATH, encoding="utf-8-sig")
df_review_raw = pd.read_csv(REVIEW_RAW_PATH, encoding="utf-8-sig")

# Simpan salinan kerja terpisah dari raw. df_news_raw / df_review_raw TIDAK
# akan diubah sepanjang notebook ini -- semua proses cleaning bekerja pada
# salinan (df_news, df_review) yang dibuat di bawah.
df_news = df_news_raw.copy()
df_review = df_review_raw.copy()

print("Dataset berita :", df_news.shape)
print("Dataset ulasan :", df_review.shape)


## TAHAP 1 — Audit Data

Audit murni bersifat *observasional*. Tidak ada baris yang dihapus atau diubah pada tahap ini.


### 1.1 Fungsi Audit Umum (dipakai ulang untuk berita & ulasan)

In [ ]:
def audit_dataframe(df: pd.DataFrame, text_col: str, name: str) -> dict:
    """
    Menghasilkan ringkasan audit umum untuk sebuah DataFrame.
    Tidak mengubah df sama sekali (read-only).

    Parameter:
        df       : DataFrame yang akan diaudit
        text_col : nama kolom teks utama (untuk statistik panjang teks)
        name     : nama dataset, hanya untuk label laporan

    Return:
        dict berisi ringkasan angka-angka audit (dipakai lagi di Tahap 6
        untuk perbandingan sebelum vs sesudah).
    """
    report = {}
    report["nama_dataset"] = name
    report["jumlah_baris"] = len(df)
    report["jumlah_kolom"] = df.shape[1]

    # Missing value per kolom
    missing = df.isna().sum()
    report["missing_value_per_kolom"] = missing[missing > 0].to_dict()
    report["total_missing_value"] = int(missing.sum())

    # Duplikasi baris penuh (semua kolom identik)
    report["jumlah_duplikasi_baris_penuh"] = int(df.duplicated().sum())

    # Statistik panjang teks (dalam karakter)
    if text_col in df.columns:
        panjang = df[text_col].fillna("").astype(str).str.len()
        report["panjang_teks_min"] = int(panjang.min())
        report["panjang_teks_max"] = int(panjang.max())
        report["panjang_teks_rata_rata"] = round(float(panjang.mean()), 2)
        report["panjang_teks_median"] = float(panjang.median())
        report["jumlah_teks_kosong"] = int((panjang == 0).sum())

    return report


def cetak_ringkasan_kolom(df: pd.DataFrame):
    """Menampilkan nama kolom, tipe data, jumlah non-null, dan contoh nilai."""
    ringkasan = pd.DataFrame({
        "kolom": df.columns,
        "tipe_data": df.dtypes.astype(str).values,
        "jumlah_non_null": df.notna().sum().values,
        "jumlah_missing": df.isna().sum().values,
        "contoh_nilai": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })
    display(ringkasan)


### 1.2 Audit Dataset Berita

In [ ]:
print("=== INFO DASAR DATASET BERITA ===")
print("Jumlah baris  :", df_news.shape[0])
print("Jumlah kolom  :", df_news.shape[1])
print()
cetak_ringkasan_kolom(df_news)


In [ ]:
print("=== RENTANG TANGGAL (mentah, sebelum divalidasi) ===")
# Coba parse tanggal hanya untuk keperluan audit; hasil parse TIDAK ditimpa ke df_news.
_tgl_coba = pd.to_datetime(df_news["Tanggal"], errors="coerce")
print("Tanggal valid terparse :", _tgl_coba.notna().sum(), "dari", len(df_news))
print("Tanggal tidak terparse :", _tgl_coba.isna().sum())
print("Rentang tanggal (yang valid) :", _tgl_coba.min(), "s/d", _tgl_coba.max())


In [ ]:
print("=== DISTRIBUSI PUBLISHER ===")
display(df_news["Publisher"].value_counts())

print()
print("=== JUMLAH NILAI UNIK KOLOM KATEGORIKAL ===")
for col in ["Publisher", "Kecamatan"]:
    print(f"{col}: {df_news[col].nunique()} nilai unik")


In [ ]:
print("=== DISTRIBUSI KECAMATAN ===")
display(df_news["Kecamatan"].value_counts())


In [ ]:
print("=== DUPLIKASI ===")
print("Duplikasi baris penuh   :", df_news.duplicated().sum())
print("Duplikasi berdasar URL  :", df_news.duplicated(subset=["URL"]).sum())
print("Duplikasi berdasar Judul:", df_news.duplicated(subset=["Judul"]).sum())

# Contoh duplikasi mendekati (judul sama tapi baris berbeda) -- hanya ditampilkan, tidak dihapus
dup_judul = df_news[df_news.duplicated(subset=["Judul"], keep=False)].sort_values("Judul")
print()
print(f"Contoh {min(5, len(dup_judul))} baris dengan judul yang duplikat/mirip:")
display(dup_judul[["Judul", "Publisher", "Tanggal", "URL"]].head(10))


In [ ]:
print("=== STATISTIK PANJANG TEKS (Isi_Berita, dalam karakter) ===")
panjang_berita = df_news["Isi_Berita"].fillna("").astype(str).str.len()
print("Minimum      :", panjang_berita.min())
print("Maksimum     :", panjang_berita.max())
print("Rata-rata    :", round(panjang_berita.mean(), 2))
print("Median       :", panjang_berita.median())
print("Teks kosong (<=1 karakter) :", (panjang_berita <= 1).sum())
print("Teks sangat pendek (<100 karakter) :", (panjang_berita < 100).sum())

print()
print("Contoh berita dengan teks sangat pendek:")
display(df_news[panjang_berita < 100][["Judul", "Isi_Berita", "URL"]].head(5))


In [ ]:
print("=== DISTRIBUSI KATA_KUNCI ===")
display(df_news["Kata_Kunci"].value_counts().head(20))


**Contoh berita yang kemungkinan TIDAK relevan dengan kesehatan (hanya untuk diperiksa manual,
belum dihapus/diberi label final di sini):**


In [ ]:
# Heuristik AUDIT SAJA: keyword lingkungan tanpa kata "kesehatan" di judul/isi.
# Ini BUKAN keputusan filtering -- hanya untuk melihat contoh kasus yang perlu
# diperiksa manual pada Tahap 5.
kata_kunci_lingkungan = df_news["Kata_Kunci"].astype(str).str.contains(
    "sampah|limbah", case=False, na=False
)
tanpa_kata_kesehatan = ~(
    df_news["Judul"].astype(str).str.contains("kesehatan|sehat|penyakit|puskesmas|rsud", case=False, na=False)
    | df_news["Isi_Berita"].astype(str).str.contains("kesehatan|sehat|penyakit|puskesmas|rsud", case=False, na=False)
)
contoh_kemungkinan_tidak_relevan = df_news[kata_kunci_lingkungan & tanpa_kata_kesehatan]

print(f"Jumlah kandidat berita 'kemungkinan tidak relevan' (heuristik audit): {len(contoh_kemungkinan_tidak_relevan)}")
display(contoh_kemungkinan_tidak_relevan[["Judul", "Kata_Kunci", "URL"]].head(10))


### 1.3 Audit Dataset Ulasan (Google Maps)

In [ ]:
print("=== INFO DASAR DATASET ULASAN ===")
print("Jumlah baris  :", df_review.shape[0])
print("Jumlah kolom  :", df_review.shape[1])
print()
cetak_ringkasan_kolom(df_review)


In [ ]:
print("=== JUMLAH TEMPAT & REVIEW PER TEMPAT ===")
print("Jumlah tempat unik (place_name):", df_review["place_name"].nunique())

review_per_tempat = df_review.groupby("place_name").size().sort_values(ascending=False)
print()
print("Distribusi jumlah review per tempat (statistik):")
display(review_per_tempat.describe())

print()
print("10 tempat dengan review terbanyak:")
display(review_per_tempat.head(10))


In [ ]:
print("=== DISTRIBUSI RATING ===")
display(df_review["rating"].value_counts(dropna=False).sort_index())
print()
print("Rating tidak valid (di luar 1-5 atau non-numerik):")
rating_numerik = pd.to_numeric(df_review["rating"], errors="coerce")
print((~rating_numerik.between(1, 5) | rating_numerik.isna()).sum(), "baris")


In [ ]:
print("=== DISTRIBUSI KATEGORI TEMPAT ===")
display(df_review["place_category"].value_counts())


In [ ]:
print("=== DISTRIBUSI is_health_related ===")
display(df_review["is_health_related"].value_counts(dropna=False))

n_health = int(df_review["is_health_related"].sum()) if df_review["is_health_related"].dtype != object \
    else (df_review["is_health_related"].astype(str).str.lower() == "true").sum()
print(f"\nJumlah review health-related: {n_health} dari {len(df_review)}")


In [ ]:
print("=== RENTANG TANGGAL REVIEW ===")
_tgl_review_coba = pd.to_datetime(df_review["review_date"], errors="coerce")
print("Tanggal valid terparse :", _tgl_review_coba.notna().sum(), "dari", len(df_review))
print("Tanggal tidak terparse :", _tgl_review_coba.isna().sum())
print("Rentang tanggal (yang valid):", _tgl_review_coba.min(), "s/d", _tgl_review_coba.max())


In [ ]:
print("=== STATISTIK PANJANG TEKS (review_text) ===")
panjang_review = df_review["review_text"].fillna("").astype(str).str.len()
print("Minimum      :", panjang_review.min())
print("Maksimum     :", panjang_review.max())
print("Rata-rata    :", round(panjang_review.mean(), 2))
print("Median       :", panjang_review.median())
print("Review kosong (0 karakter)      :", (panjang_review == 0).sum())
print("Review sangat pendek (<5 karakter):", (panjang_review < 5).sum())


In [ ]:
print("=== CONTOH REVIEW is_health_related = True ===")
mask_true = df_review["is_health_related"].astype(str).str.lower() == "true"
display(df_review[mask_true][["place_name", "review_text", "rating", "matched_keywords"]].head(5))


In [ ]:
print("=== CONTOH REVIEW is_health_related = False TETAPI punya matched_keywords ===")
mask_false_tapi_ada_keyword = (
    (df_review["is_health_related"].astype(str).str.lower() == "false")
    & df_review["matched_keywords"].notna()
    & (df_review["matched_keywords"].astype(str).str.strip() != "")
)
display(df_review[mask_false_tapi_ada_keyword][["place_name", "review_text", "matched_keywords"]].head(5))
print(f"\nTotal kasus seperti ini: {mask_false_tapi_ada_keyword.sum()} baris (perlu diperiksa manual di Tahap 5)")


In [ ]:
print("=== CONTOH REVIEW KOSONG ATAU TERLALU PENDEK ===")
display(df_review[panjang_review < 5][["place_name", "review_text", "rating", "review_date"]].head(10))


In [ ]:
print("=== DUPLIKASI DATASET ULASAN ===")
print("Duplikasi baris penuh                     :", df_review.duplicated().sum())
print("Duplikasi berdasar review_id               :", df_review.duplicated(subset=["review_id"]).sum())
print("Duplikasi berdasar (place_name, review_text):",
      df_review.duplicated(subset=["place_name", "review_text"]).sum())


## TAHAP 2 — Strategi Cleaning yang Diusulkan

Berdasarkan hasil audit di atas, berikut evaluasi tiap kandidat langkah cleaning.
**Tujuan akhir teks: input embedding SBERT**, sehingga makna semantik, struktur kalimat,
dan kata kunci kesehatan HARUS dipertahankan sebisa mungkin.

| Langkah | Terapkan? | Alasan ilmiah |
|---|---|---|
| **Lowercase** | Tidak, pada `teks_bersih` | SBERT (model transformer modern, termasuk varian IndoBERT/multilingual) sudah case-aware dan huruf kapital bisa membawa informasi (nama tempat, akronim seperti "DBD", "RSUD"). Lowercase hanya dilakukan pada kolom bantu (mis. saat pencarian keyword), bukan pada teks final. |
| **Hapus URL** | Ya | URL murni boilerplate, tidak membawa makna semantik untuk topic modeling, dan bisa mengganggu tokenisasi. |
| **Hapus HTML** | Ya | Sisa tag/entity HTML (`&nbsp;`, `<br>`) dari hasil scraping bukan bagian dari isi berita/ulasan. |
| **Hapus emoji** | Tidak dihapus, dinormalisasi | Emoji pada ulasan Google Maps bisa membawa sinyal sentimen. Karena fokus riset ini topic modeling (bukan sentimen), emoji cukup dipisahkan dari teks agar tidak mengganggu tokenizer, tapi TIDAK dihapus dari kolom asli. Diberi opsi pembersihan ringan, bukan penghapusan total. |
| **Hapus karakter aneh/non-print** | Ya (minimal) | Karakter kontrol/non-printable adalah artefak encoding, bukan konten. Tanda baca normal (titik, koma, tanda tanya) dipertahankan karena SBERT peka terhadap struktur kalimat. |
| **Normalisasi whitespace** | Ya | Spasi ganda/newline berlebih murni artefak scraping, tidak mengubah makna. |
| **Hapus boilerplate** (mis. "Baca juga:", nama reporter di akhir artikel) | Ya, dengan pola eksplisit yang dicatat | Boilerplate berulang bisa membuat BERTopic salah mengelompokkan artikel berdasarkan pola redaksi, bukan isi. Pola yang dihapus dicatat eksplisit agar bisa dijelaskan di laporan (lihat `clean_news_text`). |
| **Deduplikasi** | Ya, berbasis URL + kemiripan judul, bukan dihapus otomatis semua yang mirip | URL duplikat/hampir pasti artikel yang sama (scraping ulang). Judul mirip tapi URL beda tetap dipertahankan dan hanya diberi flag "kemungkinan_duplikat" karena bisa jadi liputan berbeda dari peristiwa yang sama. |
| **Stemming** | **Tidak** | Stemming mengubah kata ke bentuk dasar dan MERUSAK representasi yang dipelajari SBERT (SBERT dilatih pada teks natural, bukan kata berimbuhan yang dipotong). Ini pelanggaran eksplisit terhadap aturan penelitian. |
| **Stopword removal (agresif)** | **Tidak** | Sama alasannya — SBERT butuh konteks kalimat penuh; menghapus stopword mengubah struktur sintaksis yang dipakai model untuk membentuk embedding kontekstual. |
| **Hapus teks terlalu pendek** | Tidak dihapus, diberi flag | Menghapus berpotensi menghilangkan data sah (mis. ulasan singkat "puskesmas kotor" tetap informatif). Diberi flag `teks_terlalu_pendek` agar bisa diputuskan lebih lanjut sesuai kebutuhan BERTopic (BERTopic sendiri punya parameter `min_topic_size` yang bisa menangani dokumen pendek). |
| **Normalisasi tanggal** | Ya | Tanggal dengan format tidak konsisten (relatif, lokal) diseragamkan ke `YYYY-MM-DD`; baris yang gagal diparsing diberi flag, bukan dihapus. |
| **Normalisasi rating** | Ya | Rating dipaksa ke tipe numerik 1-5; nilai di luar rentang/non-numerik diberi flag `rating_tidak_valid`, nilai asli tetap disimpan di kolom `_asli`. |

**Ringkasan pembagian tanggung jawab tahap:**
- **Cleaning** (Tahap 3): hapus URL/HTML/whitespace berlebih/boilerplate, normalisasi tanggal & rating,
  deduplikasi eksplisit (URL/review_id sama persis). Tidak mengubah makna kalimat.
- **Filtering relevansi** (di luar notebook ini / tahap lanjutan setelah Tahap 5): keputusan
  "berita ini dipakai atau tidak" — SENGAJA tidak dilakukan di sini.
- **Labeling** (di luar notebook ini): pelabelan aspek/topik final, dilakukan setelah BERTopic
  (notebook 02/03), bukan berdasarkan keyword pencarian.

**Data yang dipertahankan apa adanya:** `Publisher`, `Kecamatan`, `Kata_Kunci`, `place_category`,
`place_address`, `rating`, `matched_keywords` — semua tetap ada di dataset cleaned.

**Data yang dihapus dari dataset kerja (bukan dari raw):** kolom `reviewer_name` (alasan privasi,
sesuai instruksi eksplisit — tetap ada di `df_review_raw` yang tidak dipublikasikan).

**Asumsi metodologis yang perlu ditandai:** pola boilerplate yang dihapus di `clean_news_text()`
didasarkan pada pola redaksi umum media Indonesia (mis. "Baca juga:", "ADVERTISEMENT",
"Simak Video ...") yang teramati saat audit. Jika publisher baru menggunakan pola boilerplate
lain, pola regex di fungsi tersebut perlu diperluas — ini didokumentasikan sebagai asumsi,
bukan aturan universal.


## TAHAP 3 — Kode Cleaning

Semua fungsi cleaning didefinisikan secara terpisah dan murni (*pure function*: tidak mengubah
input, mengembalikan nilai baru) agar mudah diuji dan dijelaskan satu per satu di laporan.


### 3.1 Fungsi Cleaning Teks Berita

In [ ]:
# Pola boilerplate umum media Indonesia yang teramati saat audit (Tahap 1).
# ASUMSI METODOLOGIS: daftar ini tidak lengkap untuk semua publisher, dan bisa
# diperluas jika ditemukan pola baru saat QA manual.
POLA_BOILERPLATE_BERITA = [
    r"baca juga\s*:.*", 
    r"simak juga\s*:.*",
    r"scroll untuk (melanjutkan|membaca).*",
    r"\(advertisement\)",
    r"iklan\s*$",
    r"lihat juga video.*",
]

def clean_news_text(teks: str) -> str:
    """
    Membersihkan teks berita untuk keperluan embedding SBERT.
    HANYA melakukan: hapus URL, hapus tag HTML/entity, hapus boilerplate
    redaksi yang dikenal, normalisasi whitespace, normalisasi karakter unicode.
    TIDAK melakukan lowercase, stemming, atau stopword removal.

    Parameter:
        teks : isi berita asli (string)
    Return:
        teks yang sudah dibersihkan (string). String kosong "" jika input NaN/kosong.
    """
    if pd.isna(teks):
        return ""
    teks = str(teks)

    # 1) Decode entity HTML (mis. &amp; -> &) lalu hapus sisa tag HTML
    teks = html.unescape(teks)
    teks = re.sub(r"<[^>]+>", " ", teks)

    # 2) Hapus URL (http/https maupun www.)
    teks = re.sub(r"http\S+|www\.\S+", " ", teks)

    # 3) Hapus pola boilerplate redaksi yang diketahui (case-insensitive)
    for pola in POLA_BOILERPLATE_BERITA:
        teks = re.sub(pola, " ", teks, flags=re.IGNORECASE)

    # 4) Normalisasi karakter unicode (mis. smart quotes -> quote biasa),
    #    TANPA mengubah huruf/kata itu sendiri
    teks = unicodedata.normalize("NFKC", teks)

    # 5) Hapus karakter kontrol/non-printable, tapi PERTAHANKAN tanda baca umum
    teks = "".join(ch for ch in teks if ch.isprintable() or ch in "\n\t")

    # 6) Normalisasi whitespace berlebih (spasi ganda, newline berlebih)
    teks = re.sub(r"\s+", " ", teks).strip()

    return teks


def remove_duplicate_news(df: pd.DataFrame, url_col="URL", judul_col="Judul") -> pd.DataFrame:
    """
    Menghapus duplikasi berita yang PASTI sama: URL identik.
    Judul identik dengan URL berbeda TIDAK dihapus otomatis, hanya diberi flag
    'kemungkinan_duplikat_judul' agar bisa diperiksa manual (bisa jadi liputan
    ulang dari peristiwa yang sama oleh publisher berbeda).

    Return: DataFrame baru (tidak mengubah df asli / in-place).
    """
    df = df.copy()
    df["kemungkinan_duplikat_judul"] = df.duplicated(subset=[judul_col], keep=False)

    sebelum = len(df)
    df = df.drop_duplicates(subset=[url_col], keep="first")
    sesudah = len(df)
    print(f"remove_duplicate_news: {sebelum - sesudah} baris duplikat (URL identik) dihapus "
          f"({sebelum} -> {sesudah} baris).")
    return df


def validate_dates(df: pd.DataFrame, date_col: str, new_col: str = None) -> pd.DataFrame:
    """
    Menormalisasi kolom tanggal ke format YYYY-MM-DD.
    Baris yang GAGAL diparsing TIDAK dihapus -- tanggal aslinya tetap disimpan
    di kolom '<date_col>_asli', dan kolom baru berisi NaT + flag 'tanggal_valid'.

    Return: DataFrame baru dengan kolom tambahan.
    """
    df = df.copy()
    new_col = new_col or date_col
    df[f"{date_col}_asli"] = df[date_col]
    parsed = pd.to_datetime(df[date_col], errors="coerce")
    df[new_col] = parsed.dt.strftime("%Y-%m-%d")
    df["tanggal_valid"] = parsed.notna()

    jumlah_invalid = (~df["tanggal_valid"]).sum()
    if jumlah_invalid > 0:
        print(f"validate_dates: {jumlah_invalid} baris memiliki tanggal yang tidak bisa "
              f"diparsing (kolom '{date_col}'). Baris TIDAK dihapus, hanya diberi flag "
              f"tanggal_valid=False.")
    return df

print("Fungsi cleaning berita siap: clean_news_text, remove_duplicate_news, validate_dates")


### 3.2 Fungsi Cleaning Teks Ulasan (Google Maps)

In [ ]:
def clean_review_text(teks: str) -> str:
    """
    Membersihkan teks ulasan Google Maps untuk keperluan embedding SBERT.
    Lebih ringan dibanding clean_news_text karena ulasan umumnya pendek dan
    tidak memiliki boilerplate redaksi. Emoji TIDAK dihapus, hanya diberi
    spasi pemisah agar tidak menempel pada kata (mis. 'bagus😊' -> 'bagus 😊').

    Parameter:
        teks : isi ulasan asli (string)
    Return:
        teks yang sudah dibersihkan (string). String kosong "" jika input NaN.
    """
    if pd.isna(teks):
        return ""
    teks = str(teks)

    # 1) Decode entity HTML (jarang terjadi di ulasan, tapi jaga-jaga) + hapus tag HTML
    teks = html.unescape(teks)
    teks = re.sub(r"<[^>]+>", " ", teks)

    # 2) Hapus URL jika ada (beberapa ulasan menyertakan link)
    teks = re.sub(r"http\S+|www\.\S+", " ", teks)

    # 3) Normalisasi unicode (tidak menyentuh emoji)
    teks = unicodedata.normalize("NFKC", teks)

    # 4) Beri spasi pemisah antara huruf dan emoji agar tokenisasi lebih rapi
    #    (emoji sendiri tetap dipertahankan, tidak dihapus)
    teks = re.sub(r"([\U0001F300-\U0001FAFF\u2600-\u27BF])", r" \1 ", teks)

    # 5) Normalisasi whitespace berlebih
    teks = re.sub(r"\s+", " ", teks).strip()

    return teks


def validate_reviews(df: pd.DataFrame, rating_col="rating", text_col="review_text") -> pd.DataFrame:
    """
    Validasi kolom rating dan flag review yang terlalu pendek/kosong.
    TIDAK menghapus baris apa pun -- hanya menambah kolom flag dan kolom
    rating_asli, agar keputusan pembuangan bisa diambil sadar di tahap lain.

    Return: DataFrame baru dengan kolom tambahan.
    """
    df = df.copy()

    # Normalisasi rating
    df["rating_asli"] = df[rating_col]
    rating_numerik = pd.to_numeric(df[rating_col], errors="coerce")
    df[rating_col] = rating_numerik
    df["rating_tidak_valid"] = ~rating_numerik.between(1, 5)

    # Flag teks terlalu pendek / kosong (ambang batas 5 karakter -- ASUMSI,
    # bisa disesuaikan tergantung kebutuhan BERTopic di notebook 02)
    panjang = df[text_col].fillna("").astype(str).str.len()
    df["teks_kosong"] = panjang == 0
    df["teks_terlalu_pendek"] = (panjang > 0) & (panjang < 5)

    jumlah_rating_invalid = df["rating_tidak_valid"].sum()
    jumlah_teks_pendek = df["teks_terlalu_pendek"].sum()
    print(f"validate_reviews: {jumlah_rating_invalid} rating tidak valid (flag saja, "
          f"tidak dihapus), {jumlah_teks_pendek} review terlalu pendek (flag saja).")
    return df


def remove_duplicate_reviews(df: pd.DataFrame, id_col="review_id") -> pd.DataFrame:
    """
    Menghapus duplikasi ulasan yang PASTI sama: review_id identik
    (kemungkinan hasil scraping ulang pada tempat yang sama).
    Return: DataFrame baru.
    """
    df = df.copy()
    sebelum = len(df)
    df = df.drop_duplicates(subset=[id_col], keep="first")
    sesudah = len(df)
    print(f"remove_duplicate_reviews: {sebelum - sesudah} baris duplikat (review_id identik) "
          f"dihapus ({sebelum} -> {sesudah} baris).")
    return df

print("Fungsi cleaning ulasan siap: clean_review_text, validate_reviews, remove_duplicate_reviews")


### 3.3 Terapkan Cleaning pada Dataset Berita

In [ ]:
# Simpan snapshot metrik SEBELUM cleaning (dipakai lagi di Tahap 6)
metrik_sebelum_berita = audit_dataframe(df_news, text_col="Isi_Berita", name="berita (sebelum)")

# 1) Deduplikasi berbasis URL (pasti sama), + flag duplikat judul
df_news = remove_duplicate_news(df_news, url_col="URL", judul_col="Judul")

# 2) Normalisasi tanggal (tidak menghapus baris, hanya flag tanggal_valid)
df_news = validate_dates(df_news, date_col="Tanggal")

# 3) Bersihkan teks judul & isi berita (fungsi murni, tidak mengubah makna)
df_news["judul_asli"] = df_news["Judul"]
df_news["isi_asli"] = df_news["Isi_Berita"]
df_news["teks_bersih"] = df_news["Isi_Berita"].apply(clean_news_text)

print("\nContoh sebelum vs sesudah cleaning (Isi_Berita -> teks_bersih):")
for i in df_news.index[:2]:
    print("-" * 80)
    print("SEBELUM:", str(df_news.loc[i, 'isi_asli'])[:300])
    print("SESUDAH:", str(df_news.loc[i, 'teks_bersih'])[:300])


### 3.4 Terapkan Cleaning pada Dataset Ulasan

In [ ]:
# Simpan snapshot metrik SEBELUM cleaning
metrik_sebelum_review = audit_dataframe(df_review, text_col="review_text", name="ulasan (sebelum)")

# 1) Deduplikasi berbasis review_id (pasti sama)
df_review = remove_duplicate_reviews(df_review, id_col="review_id")

# 2) Validasi rating & flag teks pendek/kosong
df_review = validate_reviews(df_review, rating_col="rating", text_col="review_text")

# 3) Bersihkan teks ulasan
df_review["review_text_asli"] = df_review["review_text"]
df_review["review_text_bersih"] = df_review["review_text"].apply(clean_review_text)

print("\nContoh sebelum vs sesudah cleaning (review_text -> review_text_bersih):")
for i in df_review.index[:3]:
    print("-" * 80)
    print("SEBELUM:", str(df_review.loc[i, 'review_text_asli'])[:200])
    print("SESUDAH:", str(df_review.loc[i, 'review_text_bersih'])[:200])


### 3.5 Pemeriksaan Hasil Cleaning (Sanity Check)

In [ ]:
# Pastikan tidak ada kolom penting yang hilang, dan tidak ada teks yang jadi
# kosong padahal aslinya tidak kosong (indikasi cleaning terlalu agresif).
def sanity_check(df_before_len, df_after, teks_asli_col, teks_bersih_col, nama):
    jadi_kosong = (
        (df_after[teks_asli_col].fillna("").astype(str).str.len() > 0)
        & (df_after[teks_bersih_col].fillna("").astype(str).str.len() == 0)
    )
    print(f"[{nama}] Baris teks asli TIDAK kosong tapi teks_bersih jadi kosong: {jadi_kosong.sum()}")
    if jadi_kosong.sum() > 0:
        print("  -> PERIKSA fungsi cleaning, kemungkinan terlalu agresif pada baris berikut:")
        display(df_after[jadi_kosong][[teks_asli_col, teks_bersih_col]].head(5))

sanity_check(len(df_news_raw), df_news, "isi_asli", "teks_bersih", "berita")
sanity_check(len(df_review_raw), df_review, "review_text_asli", "review_text_bersih", "ulasan")


## TAHAP 4 — Menyusun Dataset Hasil

Menyusun ulang kolom sesuai skema yang disepakati. `reviewer_name` DIHAPUS dari dataset
kerja/hasil (tetap ada di `df_review_raw` yang tidak disimpan ke `data/cleaned/`).


In [ ]:
# --- Dataset berita hasil ---
df_news_final = df_news[[
    "judul_asli", "isi_asli", "teks_bersih", "Publisher", "Tanggal", "Kecamatan",
    "URL", "Kata_Kunci", "kemungkinan_duplikat_judul", "tanggal_valid",
]].rename(columns={
    "Publisher": "publisher",
    "Tanggal": "tanggal",
    "Kecamatan": "kecamatan",
    "URL": "url",
    "Kata_Kunci": "kata_kunci",
})

# --- Dataset ulasan hasil (reviewer_name SENGAJA tidak disertakan) ---
df_review_final = df_review[[
    "place_name", "place_category", "place_address", "rating", "review_date",
    "review_text_asli", "review_text_bersih", "is_health_related", "matched_keywords",
    "review_id", "rating_tidak_valid", "teks_kosong", "teks_terlalu_pendek",
]]

assert "reviewer_name" not in df_review_final.columns, "reviewer_name harus dihapus dari dataset hasil!"

print("Dataset berita hasil :", df_news_final.shape)
print("Dataset ulasan hasil :", df_review_final.shape)
print("\nKolom berita:", list(df_news_final.columns))
print("Kolom ulasan:", list(df_review_final.columns))


## TAHAP 5 — Flag Kemungkinan Relevansi (Bukan Filter Final)

Kolom di bawah ini HANYA penanda untuk mempermudah pemeriksaan manual pada tahap
berikutnya (di luar notebook ini). **Tidak ada baris yang dihapus berdasarkan flag ini.**

Kategori: `kemungkinan_relevan`, `kemungkinan_tidak_relevan`, `perlu_pemeriksaan_manual`.

ASUMSI METODOLOGIS: heuristik di bawah memakai kemunculan kata kunci kesehatan di judul/isi
sebagai sinyal awal, BUKAN ground truth. Kata_Kunci hasil pencarian (mis. "Bangkalan sampah")
juga TIDAK dianggap ground truth relevansi kesehatan.


In [ ]:
KATA_KUNCI_KESEHATAN_KUAT = (
    r"kesehatan|puskesmas|rsud|rumah sakit|penyakit|dbd|demam berdarah|posyandu|"
    r"stunting|gizi buruk|imunisasi|vaksin|klinik|dokter|pandemi|wabah"
)
KATA_KUNCI_LINGKUNGAN_SAJA = r"sampah|limbah|pencemaran|banjir|sanitasi"

teks_gabungan = (
    df_news_final["judul_asli"].fillna("") + " " + df_news_final["teks_bersih"].fillna("")
)

ada_kata_kesehatan_kuat = teks_gabungan.str.contains(KATA_KUNCI_KESEHATAN_KUAT, case=False, na=False)
ada_kata_lingkungan = teks_gabungan.str.contains(KATA_KUNCI_LINGKUNGAN_SAJA, case=False, na=False)

def tentukan_flag_relevansi(row_idx):
    if ada_kata_kesehatan_kuat[row_idx]:
        return "kemungkinan_relevan"
    elif ada_kata_lingkungan[row_idx]:
        # Ada kata lingkungan tapi TIDAK ada kata kesehatan eksplisit
        # -> tidak bisa diputuskan otomatis, perlu manusia yang menilai
        return "perlu_pemeriksaan_manual"
    else:
        return "kemungkinan_tidak_relevan"

df_news_final = df_news_final.copy()
df_news_final["flag_relevansi_kesehatan"] = [
    tentukan_flag_relevansi(i) for i in range(len(df_news_final))
]

print("Distribusi flag relevansi (BUKAN keputusan final):")
display(df_news_final["flag_relevansi_kesehatan"].value_counts())


In [ ]:
print("Contoh baris 'perlu_pemeriksaan_manual' (paling butuh perhatian):")
display(df_news_final[df_news_final["flag_relevansi_kesehatan"] == "perlu_pemeriksaan_manual"]
        [["judul_asli", "kata_kunci", "url"]].head(10))


## TAHAP 6 — Quality Check: Laporan Sebelum vs Sesudah


In [ ]:
metrik_sesudah_berita = audit_dataframe(df_news_final, text_col="teks_bersih", name="berita (sesudah)")
metrik_sesudah_review = audit_dataframe(df_review_final, text_col="review_text_bersih", name="ulasan (sesudah)")

def bandingkan_metrik(sebelum: dict, sesudah: dict, metrik_list):
    baris = []
    for m in metrik_list:
        baris.append({
            "METRIK": m,
            "SEBELUM": sebelum.get(m, "-"),
            "SESUDAH": sesudah.get(m, "-"),
        })
    return pd.DataFrame(baris)

metrik_umum = [
    "jumlah_baris", "total_missing_value", "jumlah_duplikasi_baris_penuh",
    "panjang_teks_min", "panjang_teks_max", "panjang_teks_rata_rata",
    "panjang_teks_median", "jumlah_teks_kosong",
]

print("=== TABEL PERBANDINGAN: DATASET BERITA ===")
tabel_berita = bandingkan_metrik(metrik_sebelum_berita, metrik_sesudah_berita, metrik_umum)
display(tabel_berita)

print("\n=== TABEL PERBANDINGAN: DATASET ULASAN ===")
tabel_review = bandingkan_metrik(metrik_sebelum_review, metrik_sesudah_review, metrik_umum)
display(tabel_review)


In [ ]:
print("=== CONTOH SEBELUM vs SESUDAH (kualitatif) ===")
print("\n--- Berita ---")
for i in df_news_final.index[:2]:
    print("-" * 80)
    print("SEBELUM :", str(df_news.loc[i, 'isi_asli'])[:250])
    print("SESUDAH :", str(df_news_final.loc[i, 'teks_bersih'])[:250])

print("\n--- Ulasan ---")
for i in df_review_final.index[:2]:
    print("-" * 80)
    print("SEBELUM :", str(df_review_final.loc[i, 'review_text_asli'])[:200])
    print("SESUDAH :", str(df_review_final.loc[i, 'review_text_bersih'])[:200])


## TAHAP 7 — Simpan Hasil

1. Dataset raw **tidak disentuh** (tetap di `data/raw/`, tidak ditimpa notebook ini).
2. Dataset cleaned disimpan ke `data/cleaned/`.
3. Laporan audit (before/after) disimpan sebagai CSV.
4. Log langkah cleaning disimpan sebagai file teks/JSON agar bisa dilampirkan ke laporan penelitian.


In [ ]:
NEWS_CLEANED_PATH = os.path.join(CLEANED_DIR, "berita_bangkalan_cleaned.csv")
REVIEW_CLEANED_PATH = os.path.join(CLEANED_DIR, "ulasan_bangkalan_cleaned.csv")
AUDIT_REPORT_PATH = os.path.join(CLEANED_DIR, "cleaning_audit.csv")
LOG_PATH = os.path.join(CLEANED_DIR, "cleaning_log.json")

# 1) Dataset cleaned
df_news_final.to_csv(NEWS_CLEANED_PATH, index=False, encoding="utf-8-sig")
df_review_final.to_csv(REVIEW_CLEANED_PATH, index=False, encoding="utf-8-sig")

# 2) Laporan audit before/after (gabungan kedua dataset dalam satu file, mudah dibuka di Excel)
tabel_berita_export = tabel_berita.copy(); tabel_berita_export.insert(0, "dataset", "berita")
tabel_review_export = tabel_review.copy(); tabel_review_export.insert(0, "dataset", "ulasan")
tabel_audit_gabungan = pd.concat([tabel_berita_export, tabel_review_export], ignore_index=True)
tabel_audit_gabungan.to_csv(AUDIT_REPORT_PATH, index=False, encoding="utf-8-sig")

# 3) Log langkah cleaning (untuk dilampirkan ke bab metodologi laporan penelitian)
log_cleaning = {
    "tanggal_dijalankan": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "dataset_berita": {
        "baris_sebelum": metrik_sebelum_berita["jumlah_baris"],
        "baris_sesudah": metrik_sesudah_berita["jumlah_baris"],
        "langkah": [
            "remove_duplicate_news (dedup berdasarkan URL identik)",
            "validate_dates (normalisasi Tanggal -> YYYY-MM-DD, flag tanggal_valid)",
            "clean_news_text (hapus URL/HTML/boilerplate, normalisasi whitespace & unicode)",
            "flag_relevansi_kesehatan (BUKAN filter final, hanya penanda untuk pemeriksaan manual)",
        ],
    },
    "dataset_ulasan": {
        "baris_sebelum": metrik_sebelum_review["jumlah_baris"],
        "baris_sesudah": metrik_sesudah_review["jumlah_baris"],
        "langkah": [
            "remove_duplicate_reviews (dedup berdasarkan review_id identik)",
            "validate_reviews (normalisasi rating, flag teks_kosong/teks_terlalu_pendek)",
            "clean_review_text (hapus URL/HTML, normalisasi whitespace & unicode, emoji dipertahankan)",
            "reviewer_name dihapus dari dataset kerja/hasil (alasan privasi)",
        ],
    },
}
with open(LOG_PATH, "w", encoding="utf-8") as f:
    json_lib.dump(log_cleaning, f, ensure_ascii=False, indent=2)

print("Tersimpan:")
print(" -", NEWS_CLEANED_PATH)
print(" -", REVIEW_CLEANED_PATH)
print(" -", AUDIT_REPORT_PATH)
print(" -", LOG_PATH)
print("\nDataset RAW tidak diubah/ditimpa:")
print(" -", NEWS_RAW_PATH)
print(" -", REVIEW_RAW_PATH)


## Ringkasan & Rekomendasi Tahap Berikutnya

**Yang sudah dilakukan pada notebook ini:**
- Audit menyeluruh dua dataset (struktur, missing value, duplikasi, distribusi, panjang teks)
  tanpa mengubah data.
- Strategi cleaning dievaluasi satu per satu dengan alasan ilmiah, khusus untuk konteks
  SBERT + BERTopic (tidak ada stemming/stopword removal agresif).
- Cleaning modular: hapus URL/HTML/boilerplate, normalisasi whitespace/unicode/tanggal/rating,
  deduplikasi eksplisit (URL / review_id identik), tanpa mengubah makna kalimat.
- Kolom penting (tanggal, lokasi, rating, publisher, kategori) dipertahankan; `reviewer_name`
  dihapus dari dataset kerja untuk privasi.
- Flag kemungkinan relevansi dibuat sebagai penanda, bukan keputusan filtering final.
- Laporan kualitas sebelum vs sesudah dan log cleaning disimpan untuk bab metodologi.

**Potensi masalah yang masih tersisa (perlu diputuskan manusia, bukan otomatis):**
- Baris dengan flag `perlu_pemeriksaan_manual` (kata lingkungan tanpa kata kesehatan eksplisit)
  belum diputuskan relevan/tidak.
- Baris dengan `kemungkinan_duplikat_judul=True` (judul sama, URL beda) belum diverifikasi
  apakah benar-benar berita yang sama.
- Baris dengan `tanggal_valid=False` atau `rating_tidak_valid=True` masih ada di dataset dan
  perlu diputuskan (perbaiki manual / exclude saat analisis / biarkan).
- Pola boilerplate di `clean_news_text` didasarkan pada pola yang teramati saat audit ini;
  perlu di-review ulang jika publisher baru ditambahkan.

**Rekomendasi untuk `02_sbert_bertopic.ipynb`:**
1. Muat `berita_bangkalan_cleaned.csv` dan `ulasan_bangkalan_cleaned.csv` dari `data/cleaned/`.
2. Putuskan (dan dokumentasikan) bagaimana flag `flag_relevansi_kesehatan`,
   `kemungkinan_duplikat_judul`, `tanggal_valid`, `rating_tidak_valid` akan diperlakukan
   sebelum embedding (dipakai sebagai filter eksplisit dengan alasan, atau dibawa apa adanya).
3. Gunakan `teks_bersih` (berita) dan `review_text_bersih` (ulasan) sebagai input SBERT — kedua
   kolom ini sudah bersih dari boilerplate/HTML/URL namun tetap mempertahankan struktur kalimat.
4. Pertimbangkan menggabungkan kedua dataset (dengan kolom sumber `berita`/`ulasan`) atau
   memodelkan topik secara terpisah per sumber, tergantung pertanyaan penelitian di bab metodologi.
5. Simpan hasil topic modeling ke `data/processed/topic_results.csv` dan
   `data/processed/topic_summary.csv` sesuai struktur folder proyek.
